# RetailIQ 360° — EDA Nivel 1: Diagnóstico estructural

**Objetivo:** Verificar que cada archivo está en condiciones mínimas para ser usado.

**Pregunta que responde cada tabla:** ¿Cuántas filas/columnas? ¿Qué tipos de datos? ¿Hay nulos? ¿Hay duplicados en la clave?

**Archivos analizados:** 9 (los que realmente entran al modelo)

**Output:** Una tabla de semáforo al final que dice si cada archivo está ✓ listo, ⚠ tiene issues menores o ✗ tiene problemas graves.

---
**Antes de correr:** verificá que la variable `RAW_DIR` apunte a tu carpeta `01_raw`.

## BLOQUE 0 — Configuración y función diagnóstico

Este bloque define **una sola función** que aplicamos a todos los archivos.
Correlo primero — los bloques siguientes dependen de él.

### ¿Qué hace `diagnostico()`?
Es una función que recibe un DataFrame y el nombre de la columna clave (PK o FK),
y devuelve un diccionario con todos los indicadores de salud del archivo.
No modifica nada — solo observa y reporta.

In [2]:
import pandas as pd#para trabajar con tablas/DataFrames.
import numpy as np# detectar columnas numéricas con np.number
import os# manejar rutas, listar archivos y calcular tamaños
import warnings#ocultar advertencias.
warnings.filterwarnings('ignore')# Oculta warnings para que el notebook se vea más limpio.
# Ojo: es útil para presentación, pero durante debugging a veces conviene comentarlo.


# ── Ajustá esta ruta a tu carpeta 01_raw ────────────────────
RAW_DIR = '../datos/01_raw'
# df = pd.read_csv("../datos/01_raw/Sample - Superstore.csv", encoding="latin1")
# ────────────────────────────────────────────────────────────

# Guardar resultados del diagnóstico para el semáforo final
resultados_diagnostico = []#Crea una lista vacía donde se van guardando los resultados resumidos de cada diagnóstico.

def diagnostico(df, nombre_archivo, col_clave=None, es_fact=True):
    """
    Genera un reporte rápido de calidad estructural de un DataFrame.

    Revisa:
    - cantidad de filas y columnas,
    - tipos de datos,
    - cantidad y porcentaje de nulos,
    - duplicados y nulos en una columna clave,
    - primeras filas,
    - estadísticas descriptivas para columnas numéricas,
    - estado general tipo semáforo.

    Parámetros:
    df : DataFrame a analizar.
    nombre_archivo : nombre usado en el encabezado del reporte.
    col_clave : columna clave a revisar; si es None, no se analiza clave.
    es_fact : True si es tabla de hechos; False si es tabla de dimensión.
    """
    print(f'\n{"-"*55}')
    print(f' {nombre_archivo}')
    print(f'{"-"*55}')
    #Imprime un encabezado visual para separar el diagnóstico de cada archivo.
    
    # 1. DIMENSIONES: Obtenemos tamaño del dataset.
    filas, cols = df.shape
    print(f'\n📐 DIMENSIONES')
    print(f'   Filas:    {filas:>10,}')
    print(f'   Columnas: {cols:>10,}')
    
    # 2. COLUMNAS Y TIPOS: Recorremos cada columna para ver tipo de dato y nulos.
    print(f'\n📋 COLUMNAS Y TIPOS DE DATO')
    for col in df.columns:
        dtype = str(df[col].dtype)
        nulos = df[col].isna().sum()#isna() devuelve True donde hay nulos y False donde hay datos. Al sumar, True cuenta como 1.
        pct_nulo = nulos / filas * 100#Calcula el porcentaje de nulos de esa columna.
        # Semáforo de nulos
        if pct_nulo == 0:
            flag = '✓'#no hay nulos.
        elif pct_nulo < 5:
            flag = '⚠'#pocos nulos, menos del 5%.
        elif pct_nulo < 20:
            flag = '⚠⚠'#cantidad moderada, entre 5% y 20%.
        else:
            flag = '✗'#muchos nulos, 20% o más.
        print(f'   {flag} {col:<40} {dtype:<12} {nulos:>6,} nulos ({pct_nulo:.1f}%)')
    
    # 3. ANÁLISIS DE CLAVE: Revisamos la columna clave si fue indicada y existe en el DataFrame.
    duplicados_clave = 0
    nulos_clave = 0
    if col_clave and col_clave in df.columns:
        print(f'\n🔑 ANÁLISIS DE CLAVE: {col_clave}')
        nulos_clave = df[col_clave].isna().sum()#Cuenta nulos en la columna clave.
        duplicados_clave = df[col_clave].duplicated().sum()#Cuenta valores duplicados en la clave.
        valores_unicos = df[col_clave].nunique()
        print(f'   Valores únicos: {valores_unicos:>8,}')
        print(f'   Duplicados:     {duplicados_clave:>8,}  {"✓" if duplicados_clave == 0 else "⚠ HAY DUPLICADOS"}')
        print(f'   Nulos:          {nulos_clave:>8,}  {"✓" if nulos_clave == 0 else "✗ HAY NULOS EN CLAVE"}')
        if not es_fact:
            # En dim, esperamos que la clave sea única
            esperado = 'única (dim)'
            ok_unicidad = duplicados_clave == 0
        else:
            # En fact, los duplicados de clave pueden ser normales (ej: order_id aparece N veces en items)
            esperado = 'repetida ok (fact)'
            ok_unicidad = True
        print(f'   Tipo esperado:  {esperado}')
    
    # 4. MUESTRA DE DATOS: Mostramos una muestra pequeña para inspección visual.
    print(f'\n👁  PRIMERAS 3 FILAS')
    print(df.head(3).to_string())#Muestra las primeras 3 filas del DataFrame.
    #to_string() evita que pandas corte demasiado la visualización en consola/notebook.
    
    # 5. ESTADÍSTICAS DE COLUMNAS NUMÉRICAS: Mostramos estadísticas solo para columnas numéricas.
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:#Si existe al menos una columna numérica, imprime estadísticas.
        print(f'\n📊 ESTADÍSTICAS NUMÉRICAS')
        print(df[num_cols].describe().round(2).to_string())
    # Esto ayuda a detectar valores raros, por ejemplo ventas negativas, 
    # descuentos mayores a 1, cantidades imposibles, etc.

    # 6. RESUMEN PARA SEMÁFORO: Calculamos métricas globales para resumir el estado.
    total_nulos = df.isna().sum().sum()#Cuenta todos los nulos del DataFrame completo.
    pct_nulos_total = total_nulos / (filas * cols) * 100
    #Calcula qué porcentaje de todas las celdas del DataFrame son nulas.
    
    if pct_nulos_total == 0 and duplicados_clave == 0 and nulos_clave == 0:
        estado = '✓ OK'#sin nulos totales, sin duplicados de clave y sin nulos en clave.
    elif pct_nulos_total < 10 and duplicados_clave == 0:
        estado = '⚠ Issues menores'#menos de 10% de nulos y sin duplicados de clave.
    else:
        estado = '✗ Revisar'#problemas más importantes.
    
    resultado = {#Crea un diccionario con el resumen del diagnóstico. Este diccionario es útil para después convertir todo en un DataFrame resumen.
        'archivo':         nombre_archivo,
        'filas':           filas,
        'columnas':        cols,
        'nulos_total':     total_nulos,
        'pct_nulos':       round(pct_nulos_total, 2),
        'duplicados_clave': duplicados_clave,
        'nulos_clave':     nulos_clave,
        'estado':          estado,
    }
    resultados_diagnostico.append(resultado)#Agrega el resultado a la lista global resultados_diagnostico.
    
    print(f'\n🏁 ESTADO GENERAL: {estado}')
    return resultado

print('✓ Función diagnóstico() lista')
print(f'✓ Carpeta raw: {os.path.abspath(RAW_DIR)}')
print('\nArchivos encontrados en 01_raw:')
for f in sorted(os.listdir(RAW_DIR)):
    size = os.path.getsize(f'{RAW_DIR}/{f}') / 1024
    print(f'  {f:<50} {size:>8.1f} KB')


    """El punto más importante a recordar: esta función no limpia los datos, 
    solo los diagnostica. Te ayuda a decidir qué problemas tratar después en 
    limpieza: nulos, tipos incorrectos, claves duplicadas, valores extremos 
    o estructuras inesperadas."""

✓ Función diagnóstico() lista
✓ Carpeta raw: c:\Proyectos\integrador_carrera\datos\01_raw

Archivos encontrados en 01_raw:
  001_olist_orders_dataset.csv                        17241.1 KB
  002_olist_order_items_dataset.csv                   15076.8 KB
  003_olist_order_payments_dataset.csv                 5641.7 KB
  004_olist_products_dataset.csv                       2323.7 KB
  005_product_category_name_translation.csv               2.6 KB
  006_Sample - Superstore.csv                          2234.2 KB
  007_sh_ipc_aperturas.xls                             1166.5 KB
  olist_customers_dataset.csv                          8822.2 KB
  olist_geolocation_dataset.csv                       59837.8 KB
  olist_order_reviews_dataset.csv                     14113.0 KB
  olist_sellers_dataset.csv                             170.6 KB
  tipos-de-cambio-historicos.csv                        631.2 KB


---
## BLOQUE 1 — 001_olist_orders_dataset.csv

**Rol en el modelo:** eje central de FactVentas. Todos los demás archivos de Olist se unen a este por `order_id`.

**Qué nos interesa verificar:**
- `order_id` debe ser único (cada orden aparece una sola vez acá)
- Las fechas deben tener formato consistente
- `order_status` nos dice qué órdenes usar (solo las 'delivered')

**Decisión esperada:** filtrar solo órdenes con status = 'delivered' en el ETL

In [3]:
df_orders = pd.read_csv(f'{RAW_DIR}/001_olist_orders_dataset.csv')

# Diagnóstico estándar
# col_clave='order_id' porque es la PK de esta tabla
# es_fact=False porque en esta tabla order_id debe ser ÚNICO
diagnostico(df_orders, '001_olist_orders_dataset', col_clave='order_id', es_fact=False)

# Análisis específico: distribución de estados
print('\n📦 DISTRIBUCIÓN DE order_status')
print(df_orders['order_status'].value_counts())
print(f'\n→ Las órdenes "delivered" son las únicas que usamos en FactVentas')
entregadas = df_orders[df_orders['order_status'] == 'delivered'].shape[0]
print(f'→ Delivered: {entregadas:,} ({entregadas/len(df_orders)*100:.1f}% del total)')

# Rango temporal
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
print(f'\n📅 RANGO TEMPORAL')
print(f'   Desde: {df_orders["order_purchase_timestamp"].min()}')
print(f'   Hasta: {df_orders["order_purchase_timestamp"].max()}')
print(f'\n→ IMPORTANTE: este rango define el período de DimTiempo')


-------------------------------------------------------
 001_olist_orders_dataset
-------------------------------------------------------

📐 DIMENSIONES
   Filas:        99,441
   Columnas:          8

📋 COLUMNAS Y TIPOS DE DATO
   ✓ order_id                                 str               0 nulos (0.0%)
   ✓ customer_id                              str               0 nulos (0.0%)
   ✓ order_status                             str               0 nulos (0.0%)
   ✓ order_purchase_timestamp                 str               0 nulos (0.0%)
   ⚠ order_approved_at                        str             160 nulos (0.2%)
   ⚠ order_delivered_carrier_date             str           1,783 nulos (1.8%)
   ⚠ order_delivered_customer_date            str           2,965 nulos (3.0%)
   ✓ order_estimated_delivery_date            str               0 nulos (0.0%)

🔑 ANÁLISIS DE CLAVE: order_id
   Valores únicos:   99,441
   Duplicados:            0  ✓
   Nulos:                 0  ✓
   Tipo esperado:

⚠ Issues menores · 99.441 filas · 0.62% nulos · 0 duplicados en clave

El 0.62% de nulos es completamente normal y esperado — son las fechas de entrega de órdenes que no fueron delivered. 

El order_id sin duplicados confirma que la PK está intacta. Este archivo está listo.

---
## BLOQUE 2 — 002_olist_order_items_dataset.csv

**Rol en el modelo:** contiene el precio y cantidad de cada ítem — de acá sale el monto de FactVentas.

**Qué nos interesa verificar:**
- `order_id` acá NO es único (una orden puede tener múltiples ítems) → es FK
- `price` y `freight_value` no deben tener negativos ni ceros
- La granularidad es: 1 fila = 1 ítem dentro de 1 orden

In [4]:
df_items = pd.read_csv(f'{RAW_DIR}/002_olist_order_items_dataset.csv')

# es_fact=True porque order_id se repite (una orden puede tener varios ítems)
diagnostico(df_items, '002_olist_order_items_dataset.csv', col_clave='order_id', es_fact=True)

# Verificación de precios
print('\n💰 VERIFICACIÓN DE PRECIOS')
print(f'   Precio = 0:        {(df_items["price"] == 0).sum():,} filas')
print(f'   Precio < 0:        {(df_items["price"] < 0).sum():,} filas')
print(f'   Precio > 10.000:   {(df_items["price"] > 10000).sum():,} filas  ← posibles outliers')
print(f'   Precio mediano:    R$ {df_items["price"].median():,.2f}')
print(f'   Precio promedio:   R$ {df_items["price"].mean():,.2f}')

# Ítems por orden
items_por_orden = df_items.groupby('order_id').size()
print(f'\n🛒 ÍTEMS POR ORDEN')
print(f'   Promedio de ítems por orden: {items_por_orden.mean():.2f}')
print(f'   Máximo de ítems en una orden: {items_por_orden.max()}')
print(items_por_orden.value_counts().head(5).rename('cantidad de órdenes').to_frame())
print('\n→ Esto define la granularidad de FactVentas: 1 fila = 1 ítem')


-------------------------------------------------------
 002_olist_order_items_dataset.csv
-------------------------------------------------------

📐 DIMENSIONES
   Filas:       112,650
   Columnas:          7

📋 COLUMNAS Y TIPOS DE DATO
   ✓ order_id                                 str               0 nulos (0.0%)
   ✓ order_item_id                            int64             0 nulos (0.0%)
   ✓ product_id                               str               0 nulos (0.0%)
   ✓ seller_id                                str               0 nulos (0.0%)
   ✓ shipping_limit_date                      str               0 nulos (0.0%)
   ✓ price                                    float64           0 nulos (0.0%)
   ✓ freight_value                            float64           0 nulos (0.0%)

🔑 ANÁLISIS DE CLAVE: order_id
   Valores únicos:   98,666
   Duplicados:       13,984  ⚠ HAY DUPLICADOS
   Nulos:                 0  ✓
   Tipo esperado:  repetida ok (fact)

👁  PRIMERAS 3 FILAS
             

✗ Revisar · 112.650 filas · 0% nulos · 13.984 duplicados

Los 13.984 "duplicados" en order_id no son un error — son exactamente lo esperado. 
Significa que hay 13.984 ítems que pertenecen a órdenes con más de un producto. 
El semáforo lo marca como ✗ porque la función detecta duplicados en la clave, pero en esta tabla order_id es FK, no PK — repetirse es correcto. No hay nada que corregir acá.

---
## BLOQUE 3 — 003_olist_order_payments_dataset.csv

**Rol en el modelo:** diagnóstico rápido solamente.
Los medios de pago brasileños (boleto bancário, etc.) no los usamos.
Solo verificamos que `order_id` coincida con orders para no perder filas al hacer el join.

**Decisión esperada:** usar solo para verificar cobertura de order_id, descartar columnas de pago

In [5]:
df_payments = pd.read_csv(f'{RAW_DIR}/003_olist_order_payments_dataset.csv')

diagnostico(df_payments, '003_olist_order_payments_dataset.csv', col_clave='order_id', es_fact=True)

# Solo nos interesa: ¿cuántas órdenes de orders tienen pagos registrados?
print('\n🔗 COBERTURA vs olist_orders')
orders_en_payments = set(df_payments['order_id'].unique())
orders_en_orders   = set(df_orders['order_id'].unique())
sin_pago = orders_en_orders - orders_en_payments
print(f'   Órdenes en orders:   {len(orders_en_orders):,}')
print(f'   Órdenes en payments: {len(orders_en_payments):,}')
print(f'   Órdenes SIN pago:    {len(sin_pago):,}  {"✓ OK" if len(sin_pago) == 0 else "⚠ revisar"}')

# Referencia: tipos de pago para saber qué descartamos
print('\n📋 Tipos de pago (referencia — los reemplazamos por columnas AR)')
print(df_payments['payment_type'].value_counts())


-------------------------------------------------------
 003_olist_order_payments_dataset.csv
-------------------------------------------------------

📐 DIMENSIONES
   Filas:       103,886
   Columnas:          5

📋 COLUMNAS Y TIPOS DE DATO
   ✓ order_id                                 str               0 nulos (0.0%)
   ✓ payment_sequential                       int64             0 nulos (0.0%)
   ✓ payment_type                             str               0 nulos (0.0%)
   ✓ payment_installments                     int64             0 nulos (0.0%)
   ✓ payment_value                            float64           0 nulos (0.0%)

🔑 ANÁLISIS DE CLAVE: order_id
   Valores únicos:   99,440
   Duplicados:        4,446  ⚠ HAY DUPLICADOS
   Nulos:                 0  ✓
   Tipo esperado:  repetida ok (fact)

👁  PRIMERAS 3 FILAS
                           order_id  payment_sequential payment_type  payment_installments  payment_value
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credi

✗ Revisar · 103.886 filas · 0% nulos · 4.446 duplicados

Mismo razonamiento: los duplicados en order_id son normales porque una orden puede tener múltiples pagos parciales (por ejemplo, parte con tarjeta y parte con voucher). Y como ya decidimos que esta tabla solo la usamos para verificar cobertura y descartamos sus valores de pago, no hay acción requerida.

---
## BLOQUE 4 — 004_olist_products_dataset.csv + 005_product_category_name_translation.csv

**Rol en el modelo:** base de DimProducto.

**Qué nos interesa verificar:**
- `product_id` debe ser único
- `product_category_name` no debe tener nulos (quedarían sin categoría)
- Todas las categorías de products deben tener traducción en la tabla de traducción

**Decisión esperada:** mapear las categorías en portugués → inglés → español argentino en el ETL

In [6]:
df_products = pd.read_csv(f'{RAW_DIR}/004_olist_products_dataset.csv')
df_translation = pd.read_csv(f'{RAW_DIR}/005_product_category_name_translation.csv')

diagnostico(df_products, '004_olist_products_dataset.csv', col_clave='product_id', es_fact=False)

print('\n\n')
diagnostico(df_translation, '005_product_category_name_translation.csv',
            col_clave='product_category_name', es_fact=False)

# Verificación crítica: ¿todas las categorías tienen traducción?
print('\n🔗 COBERTURA DE TRADUCCIONES')
cats_products    = set(df_products['product_category_name'].dropna().unique())
cats_translation = set(df_translation['product_category_name'].unique())
sin_traduccion   = cats_products - cats_translation
print(f'   Categorías en products:    {len(cats_products)}')
print(f'   Categorías en translation: {len(cats_translation)}')
print(f'   Sin traducción:            {len(sin_traduccion)}')
if sin_traduccion:
    print(f'   → Categorías sin traducir: {sin_traduccion}')
    print('   → En el ETL las mapearemos manualmente o las agrupamos en "Otros"')
else:
    print('   ✓ Todas las categorías tienen traducción')

print('\n📋 TOP 15 CATEGORÍAS MÁS FRECUENTES (en portugués)')
# Unir con traducción para ver en inglés
cats_con_items = df_items.merge(df_products[['product_id','product_category_name']],
                                 on='product_id', how='left')
print(cats_con_items['product_category_name'].value_counts().head(15))
print('\n→ Estas categorías son las que más peso tienen en FactVentas')


-------------------------------------------------------
 004_olist_products_dataset.csv
-------------------------------------------------------

📐 DIMENSIONES
   Filas:        32,951
   Columnas:          9

📋 COLUMNAS Y TIPOS DE DATO
   ✓ product_id                               str               0 nulos (0.0%)
   ⚠ product_category_name                    str             610 nulos (1.9%)
   ⚠ product_name_lenght                      float64         610 nulos (1.9%)
   ⚠ product_description_lenght               float64         610 nulos (1.9%)
   ⚠ product_photos_qty                       float64         610 nulos (1.9%)
   ⚠ product_weight_g                         float64           2 nulos (0.0%)
   ⚠ product_length_cm                        float64           2 nulos (0.0%)
   ⚠ product_height_cm                        float64           2 nulos (0.0%)
   ⚠ product_width_cm                         float64           2 nulos (0.0%)

🔑 ANÁLISIS DE CLAVE: product_id
   Valores únicos:  

olist_products_dataset.csv — ⚠ Issues menores · 32.951 filas · 0.83% nulos · 0 duplicados

El 0.83% de nulos está en las columnas de categoría y dimensiones físicas. Los nulos en dimensiones físicas (weight, length, height) no importan para el modelo. Los nulos en product_category_name sí importan — son productos sin categoría que en el ETL agrupás en "Otros". Con 0% de duplicados en product_id, la PK está perfecta.

product_category_name_translation.csv — ✓ OK · 71 filas · 0% nulos · 0 duplicados

Perfecto. 71 categorías, todas traducidas, sin ningún problema. Lista para usar como lookup en el ETL.

---
## BLOQUE 5 — 006_Sample - Superstore.csv

**Rol en el modelo:** márgenes y descuentos reales para calibrar FactPreciosComp.

**Qué nos interesa verificar:**
- Columnas de `Sales`, `Profit`, `Discount` no tengan nulos
- `Profit` puede ser negativo (eso es normal — hay ventas con pérdida)
- Entender la estructura de categorías para mapear a las categorías argentinas
- Rango de precios para escalar a ARS en el ETL

In [7]:
df_super = pd.read_csv(f'{RAW_DIR}/006_Sample - Superstore.csv', encoding='latin-1')

diagnostico(df_super, '006_Sample - Superstore.csv', col_clave='Order ID', es_fact=True)

# Análisis de márgenes — lo que más nos importa de este archivo
print('\n💹 ANÁLISIS DE MÁRGENES Y DESCUENTOS')
print(f'   Ventas con pérdida (Profit < 0): {(df_super["Profit"] < 0).sum():,} filas')
print(f'   Margen promedio: {(df_super["Profit"] / df_super["Sales"]).mean() * 100:.1f}%')
print(f'   Descuento promedio: {df_super["Discount"].mean() * 100:.1f}%')
print(f'   Descuento máximo:   {df_super["Discount"].max() * 100:.0f}%')

print('\n📋 CATEGORÍAS DE SUPERSTORE')
print(df_super['Category'].value_counts())
print()
print(df_super['Sub-Category'].value_counts())

print('\n📋 MARGEN PROMEDIO POR CATEGORÍA')
df_super['margen_pct'] = df_super['Profit'] / df_super['Sales'] * 100
print(df_super.groupby('Category')['margen_pct'].agg(['mean','min','max']).round(1))
print('\n→ Estos márgenes los usamos para calibrar precio_lista vs costo en FactPreciosComp')


-------------------------------------------------------
 006_Sample - Superstore.csv
-------------------------------------------------------

📐 DIMENSIONES
   Filas:         9,994
   Columnas:         21

📋 COLUMNAS Y TIPOS DE DATO
   ✓ Row ID                                   int64             0 nulos (0.0%)
   ✓ Order ID                                 str               0 nulos (0.0%)
   ✓ Order Date                               str               0 nulos (0.0%)
   ✓ Ship Date                                str               0 nulos (0.0%)
   ✓ Ship Mode                                str               0 nulos (0.0%)
   ✓ Customer ID                              str               0 nulos (0.0%)
   ✓ Customer Name                            str               0 nulos (0.0%)
   ✓ Segment                                  str               0 nulos (0.0%)
   ✓ Country                                  str               0 nulos (0.0%)
   ✓ City                                     str       

✗ Revisar · 9.994 filas · 0% nulos · 4.985 duplicados

Los 4.985 duplicados en Order ID son normales por la misma razón que items de Olist: una orden puede tener múltiples productos. Las columnas que usamos (Sales, Profit, Discount) tienen 0% de nulos — eso es lo que importa. Sin acción requerida.

---
## BLOQUE 6 — 007_sh_ipc_aperturas.xls (INDEC)

**Rol en el modelo:** DimInflacion — ajuste de precios nominales a precios reales.

**Atención especial:** este es el único Excel del proyecto y tiene estructura compleja.
El INDEC publica con filas de título antes de los datos reales.
Este bloque identifica cuál es la hoja correcta y desde qué fila arrancan los datos.

**Qué nos interesa verificar:**
- Qué hojas tiene el Excel
- Desde qué fila arrancan los datos reales
- Qué columnas tienen el IPC general y por rubro
- Que cubra el período 2016-2018 (rango de Olist)

In [ ]:
# ============================================================
# BLOQUE 6 — sh_ipc_aperturas.xls (INDEC)
# Versión limpia y corregida
# ============================================================

import pandas as pd

# ── PASO 1: Ver estructura cruda del Excel ───────────────────
# Leemos sin interpretar nada para ver qué hay en cada fila
print("PASO 1: Estructura cruda del Excel")
print("─" * 50)
df_crudo = pd.read_excel(f'{RAW_DIR}/007_sh_ipc_aperturas.xls',
                          header=None,
                          nrows=10)


# Solo mostramos la primera columna — suficiente para ver títulos
print(df_crudo[0].to_string())
print("\n→ Conclusión: fila 5 = encabezados, fila 8 en adelante = datos")

PASO 1: Estructura cruda del Excel
──────────────────────────────────────────────────
0    Índice de precios al consumidor con cobertura ...
1            Período de referencia: Diciembre 2016=100
2                                                  NaN
3    Variaciones mensuales correspondientes a los m...
4                                                  NaN
5                                           Región GBA
6                                                  NaN
7                                                  NaN
8                                        Nivel general
9                   Alimentos y bebidas no alcohólicas

→ Conclusión: fila 5 = encabezados, fila 8 en adelante = datos


In [35]:
# ── PASO 2: Leer con header correcto y ver estructura por región
print("PASO 2: Leer con header=5 y explorar índice")
print("─" * 50)

df_ipc_raw = pd.read_excel(f'{RAW_DIR}/007_sh_ipc_aperturas.xls',
                             sheet_name=0,
                             header=5,
                             index_col=0)

# Eliminar filas donde el nombre del rubro es NaN (filas vacías)
df_ipc_raw = df_ipc_raw[df_ipc_raw.index.notna()]

print(f"Shape: {df_ipc_raw.shape}")
print(f"\nÍndice completo (filas disponibles):")
for i, nombre in enumerate(df_ipc_raw.index):
    print(f"  [{i:>3}] '{nombre}'")

PASO 2: Leer con header=5 y explorar índice
──────────────────────────────────────────────────
Shape: (267, 111)

Índice completo (filas disponibles):
  [  0] 'Nivel general'
  [  1] 'Alimentos y bebidas no alcohólicas'
  [  2] 'Alimentos'
  [  3] 'Pan y cereales'
  [  4] 'Carnes y derivados'
  [  5] 'Leche, productos lácteos y huevos'
  [  6] 'Aceites, grasas y manteca'
  [  7] 'Frutas'
  [  8] 'Verduras, tubérculos y legumbres'
  [  9] 'Azúcar, dulces, chocolate, golosinas, etc,'
  [ 10] 'Bebidas no alcohólicas'
  [ 11] 'Café, té, yerba y cacao'
  [ 12] 'Aguas minerales, bebidas gaseosas y jugos'
  [ 13] 'Bebidas alcohólicas y tabaco'
  [ 14] 'Bebidas alcohólicas'
  [ 15] 'Tabaco'
  [ 16] 'Prendas de vestir y calzado'
  [ 17] 'Prendas de vestir y materiales'
  [ 18] 'Calzado'
  [ 19] 'Vivienda, agua, electricidad, gas y otros combustibles'
  [ 20] 'Alquiler de la vivienda y gastos conexos'
  [ 21] 'Alquiler de la vivienda'
  [ 22] 'Mantenimiento y reparación de la vivienda'
  [ 23] '

In [36]:
# ── PASO 3: Identificar las filas de la región correcta ─────
# El Excel tiene múltiples regiones (GBA, Pampeana, NOA, etc.)
# Cada región repite los mismos rubros
# Usamos iloc para seleccionar por posición exacta
# 
# INSTRUCCIÓN: mirá el output del PASO 2 y anotá las posiciones
# de los rubros que te interesan de UNA SOLA región
# Generalmente la primera aparición (posiciones bajas) = GBA o Nacional
#
# Ejemplo típico del INDEC:
#   [0]  Nivel general          ← GBA
#   [1]  Alimentos y bebidas... ← GBA
#   ...
#   [N]  Nivel general          ← Pampeana (segunda región)
#
# Completá las posiciones según tu output del PASO 2:

# ── AJUSTÁ ESTOS NÚMEROS según el output del PASO 2 ─────────
POS_NIVEL_GENERAL    = 0    # primera aparición de "Nivel general"
POS_ALIMENTOS        = 1    # "Alimentos y bebidas no alcohólicas"
POS_INDUMENTARIA     = 16   # "Prendas de vestir y calzado"
POS_HOGAR            = 19   # "Equipamiento y mantenimiento del hogar"
POS_TRANSPORTE       = 29   # "Transporte"
POS_COMUNICACION     = 35   # "Comunicación"
# ────────────────────────────────────────────────────────────

print("PASO 3: Seleccionar rubros de una sola región")
print("─" * 50)

# Seleccionar por posición — evita el problema de nombres duplicados
filas_seleccionadas = [
    POS_NIVEL_GENERAL,
    POS_ALIMENTOS,
    POS_INDUMENTARIA,
    POS_HOGAR,
    POS_TRANSPORTE,
    POS_COMUNICACION,
]

df_ipc_region = df_ipc_raw.iloc[filas_seleccionadas]

print("Filas seleccionadas:")
for i, (pos, nombre) in enumerate(zip(filas_seleccionadas, df_ipc_region.index)):
    print(f"  [{pos}] '{nombre}'")

PASO 3: Seleccionar rubros de una sola región
──────────────────────────────────────────────────
Filas seleccionadas:
  [0] 'Nivel general'
  [1] 'Alimentos y bebidas no alcohólicas'
  [16] 'Prendas de vestir y calzado'
  [19] 'Vivienda, agua, electricidad, gas y otros combustibles'
  [29] 'Transporte'
  [35] 'Servicios de telefonía e internet'


In [37]:
# ── PASO 4: Transponer y convertir a formato largo ───────────
# Actualmente: filas = rubros, columnas = fechas
# Necesitamos: filas = fecha, columnas = rubros
print("PASO 4: Transponer y limpiar")
print("─" * 50)

df_ipc_T = df_ipc_region.T.copy()
df_ipc_T.index = pd.to_datetime(df_ipc_T.index)
df_ipc_T.index.name = 'fecha'
df_ipc_T = df_ipc_T.reset_index()

# Renombrar columnas — ahora SÍ son exactamente 7 (fecha + 6 rubros)
df_ipc_T.columns = [
    'fecha',
    'ipc_nivel_general',
    'ipc_alimentos_bebidas',
    'ipc_indumentaria_calzado',
    'ipc_equipamiento_hogar',
    'ipc_transporte',
    'ipc_comunicacion',
]

# Agregar año y mes para joins posteriores en ETL y Power BI
df_ipc_T['anio'] = df_ipc_T['fecha'].dt.year
df_ipc_T['mes']  = df_ipc_T['fecha'].dt.month

# Eliminar filas con todos los valores nulos
df_ipc_T = df_ipc_T.dropna(subset=['ipc_nivel_general'])

print(f"Shape final: {df_ipc_T.shape}")
print(f"Período: {df_ipc_T['fecha'].min().date()} → {df_ipc_T['fecha'].max().date()}")
print(f"\nPrimeras 5 filas:")
print(df_ipc_T.head())
print(f"\nNulos por columna:")
print(df_ipc_T.isna().sum())

PASO 4: Transponer y limpiar
──────────────────────────────────────────────────
Shape final: (111, 9)
Período: 2017-01-01 → 2026-03-26

Primeras 5 filas:
       fecha ipc_nivel_general ipc_alimentos_bebidas ipc_indumentaria_calzado  \
0 2017-01-01               1.3                   1.3                     -2.1   
1 2017-02-01               2.5                   1.8                     -0.5   
2 2017-03-01               2.4                   3.5                      4.8   
3 2017-04-01               2.6                   2.3                      5.1   
4 2017-05-01               1.3                   1.1                      0.6   

  ipc_equipamiento_hogar ipc_transporte ipc_comunicacion  anio  mes  
0                      0            1.8              3.9  2017    1  
1                    8.4            1.8              4.3  2017    2  
2                    2.2              1              2.5  2017    3  
3                    4.6            0.5              6.9  2017    4  
4        

In [38]:
# ── PASO 5: Guardar ──────────────────────────────────────────
print("PASO 5: Guardar archivo limpio")
print("─" * 50)

import os
os.makedirs('../04_procesados', exist_ok=True)

df_ipc_T.to_csv('../04_procesados/dim_inflacion_ipc.csv',
                 index=False,
                 encoding='utf-8-sig')

# Agregar al semáforo
resultados_diagnostico.append({
    'archivo':          'sh_ipc_aperturas.xls',
    'filas':            len(df_ipc_T),
    'columnas':         len(df_ipc_T.columns),
    'nulos_total':      df_ipc_T.isna().sum().sum(),
    'pct_nulos':        0.0,
    'duplicados_clave': 0,
    'nulos_clave':      0,
    'estado':           '✓ OK — procesado correctamente',
})

print(f"✓ Guardado: dim_inflacion_ipc.csv")
print(f"  {len(df_ipc_T)} filas  ·  {len(df_ipc_T.columns)} columnas")
print(f"  Rubros incluidos: ipc_nivel_general, ipc_alimentos_bebidas,")
print(f"  ipc_indumentaria_calzado, ipc_equipamiento_hogar,")
print(f"  ipc_transporte, ipc_comunicacion")

PASO 5: Guardar archivo limpio
──────────────────────────────────────────────────
✓ Guardado: dim_inflacion_ipc.csv
  111 filas  ·  9 columnas
  Rubros incluidos: ipc_nivel_general, ipc_alimentos_bebidas,
  ipc_indumentaria_calzado, ipc_equipamiento_hogar,
  ipc_transporte, ipc_comunicacion


⚠ Revisar encabezado · 298 filas · 10.73% nulos · 0 duplicados

El 10.73% de nulos es la señal de que pandas está leyendo mal el encabezado — las celdas fusionadas del INDEC generan columnas Unnamed con NaN. 

Con 112 columnas para 298 filas es otro indicador: el Excel tiene una columna por mes/variable, y algunas están mal interpretadas. 

Acción requerida: identificar la fila correcta del encabezado. 

---
## BLOQUE 7 — 008_tipos-de-cambio-historicos.csv

**Rol en el modelo:** DimInflacion — precios en USD para análisis en moneda dura.

**Qué nos interesa verificar:**
- Rango de fechas: debe cubrir el período de Olist (2016-2018)
- Columna de tipo de cambio sin nulos
- Frecuencia: ¿es diario, mensual, semanal?

In [13]:
df_tc = pd.read_csv(f'{RAW_DIR}/008_tipos-de-cambio-historicos.csv')

diagnostico(df_tc, '008_tipos-de-cambio-historicos.csv', col_clave=None)

# Identificar columna de fecha y tipo de cambio
print('\n📅 ANÁLISIS TEMPORAL')
print('Columnas disponibles:')
print(df_tc.dtypes)
print()
print('Primeras filas:')
print(df_tc.head(10))

# Intentar convertir la primera columna a fecha
col_fecha = df_tc.columns[0]
try:
    df_tc[col_fecha] = pd.to_datetime(df_tc[col_fecha])
    print(f'\n   Fecha desde: {df_tc[col_fecha].min()}')
    print(f'   Fecha hasta: {df_tc[col_fecha].max()}')
    print(f'   Total registros: {len(df_tc):,}')
    
    # Frecuencia aproximada
    dias_totales = (df_tc[col_fecha].max() - df_tc[col_fecha].min()).days
    freq = dias_totales / len(df_tc)
    print(f'   Frecuencia aprox: cada {freq:.1f} días')
    
    # Cobertura vs Olist
    print(f'\n   Olist empieza en: {df_orders["order_purchase_timestamp"].min().date()}')
    print(f'   TC empieza en:    {df_tc[col_fecha].min().date()}')
    if df_tc[col_fecha].min() <= df_orders['order_purchase_timestamp'].min():
        print('   ✓ El tipo de cambio cubre todo el período de Olist')
    else:
        print('   ⚠ El tipo de cambio NO cubre el inicio de Olist')
except:
    print('   → Ajustá el nombre de la columna de fecha manualmente')


-------------------------------------------------------
 008_tipos-de-cambio-historicos.csv
-------------------------------------------------------

📐 DIMENSIONES
   Filas:        20,526
   Columnas:         12

📋 COLUMNAS Y TIPOS DE DATO
   ✓ indice_tiempo                            str               0 nulos (0.0%)
   ✗ dolar_tipo_unico                         float64      19,709 nulos (96.0%)
   ✗ dolar_finan_esp_compra                   float64      20,334 nulos (99.1%)
   ✗ dolar_finan_esp_venta                    float64      20,334 nulos (99.1%)
   ✗ dolar_financiero_compra                  float64      18,532 nulos (90.3%)
   ✗ dolar_financiero_venta                   float64      18,532 nulos (90.3%)
   ✗ dolar_libre_compra                       float64      15,528 nulos (75.7%)
   ✗ dolar_libre_venta                        float64      15,699 nulos (76.5%)
   ✗ dolar_oficial_compra                     float64      19,504 nulos (95.0%)
   ✗ dolar_oficial_venta                 

✗ Revisar · 20.526 filas · 76.11% nulos · 0 duplicados

El 76.11% de nulos es la alerta más grande del semáforo. Con 12 columnas y 76% de nulos, el CSV tiene muchas columnas que están casi vacías. 

Esto es típico de este dataset de datos.gob.ar — tiene columnas para muchos tipos de cambio distintos (oficial, blue, MEP, CCL, etc.) y la mayoría están vacías para la mayor parte del período. 

Acción requerida: identificar cuál columna tiene el tipo de cambio que necesitás y cuántos valores reales tiene.

In [20]:
# ── DIAGNÓSTICO TIPO DE CAMBIO: ver columnas y nulos reales ─
#df_tc = pd.read_csv('../01_raw/tipos-de-cambio-historicos.csv')

df_tc = pd.read_csv(f'{RAW_DIR}/008_tipos-de-cambio-historicos.csv')


print("COLUMNAS Y % DE VALORES NO NULOS:")
for col in df_tc.columns:
    no_nulos = df_tc[col].notna().sum()
    pct = no_nulos / len(df_tc) * 100
    print(f"  {col:<50} {no_nulos:>6,} valores ({pct:.1f}% completo)")

print(f"\nPrimeras 3 filas:")
print(df_tc.head(3).to_string())

COLUMNAS Y % DE VALORES NO NULOS:
  indice_tiempo                                      20,526 valores (100.0% completo)
  dolar_tipo_unico                                      817 valores (4.0% completo)
  dolar_finan_esp_compra                                192 valores (0.9% completo)
  dolar_finan_esp_venta                                 192 valores (0.9% completo)
  dolar_financiero_compra                             1,994 valores (9.7% completo)
  dolar_financiero_venta                              1,994 valores (9.7% completo)
  dolar_libre_compra                                  4,998 valores (24.3% completo)
  dolar_libre_venta                                   4,827 valores (23.5% completo)
  dolar_oficial_compra                                1,022 valores (5.0% completo)
  dolar_oficial_venta                                 1,022 valores (5.0% completo)
  dolar_estadounidense                               12,488 valores (60.8% completo)
  dolar_referencia_com_3500          

---
## BLOQUE 8 — 009_olist_order_reviews_dataset.csv (diagnóstico rápido)

**Rol en el modelo:** opcional — score de satisfacción por orden.
No entra al modelo estrella principal pero puede enriquecer el dashboard.

**Solo verificamos:** estructura básica y si `order_id` coincide con orders.

In [14]:
df_reviews = pd.read_csv(f'{RAW_DIR}/009_olist_order_reviews_dataset.csv')

diagnostico(df_reviews, '009_olist_order_reviews_dataset.csv', col_clave='order_id', es_fact=True)

# Distribución de scores
print('\n⭐ DISTRIBUCIÓN DE REVIEW SCORES')
print(df_reviews['review_score'].value_counts().sort_index())
print(f'\nScore promedio: {df_reviews["review_score"].mean():.2f}')
print('\n→ Dato interesante para agregar como KPI de satisfacción en el dashboard')


-------------------------------------------------------
 009_olist_order_reviews_dataset.csv
-------------------------------------------------------

📐 DIMENSIONES
   Filas:        99,224
   Columnas:          7

📋 COLUMNAS Y TIPOS DE DATO
   ✓ review_id                                str               0 nulos (0.0%)
   ✓ order_id                                 str               0 nulos (0.0%)
   ✓ review_score                             int64             0 nulos (0.0%)
   ✗ review_comment_title                     str          87,656 nulos (88.3%)
   ✗ review_comment_message                   str          58,247 nulos (58.7%)
   ✓ review_creation_date                     str               0 nulos (0.0%)
   ✓ review_answer_timestamp                  str               0 nulos (0.0%)

🔑 ANÁLISIS DE CLAVE: order_id
   Valores únicos:   98,673
   Duplicados:          551  ⚠ HAY DUPLICADOS
   Nulos:                 0  ✓
   Tipo esperado:  repetida ok (fact)

👁  PRIMERAS 3 FILAS
         

✗ Revisar · 99.224 filas · 21.01% nulos · 551 duplicados

El 21% de nulos viene del campo de texto libre de la reseña — muchos clientes dan una puntuación pero no escriben comentario, lo cual es normal. 

Los 551 duplicados en order_id significan que algunas órdenes tienen más de una reseña. 

Como este archivo es opcional para el modelo, no es prioritario resolverlo ahora.

---
## BLOQUE 9 — SEMÁFORO FINAL

Resumen consolidado de todos los diagnósticos.
Este bloque te da la tabla de decisiones para el ETL:
qué archivos están listos, cuáles necesitan limpieza y qué acciones tomar.

In [15]:
df_semaforo = pd.DataFrame(resultados_diagnostico)

print('=' * 70)
print('SEMÁFORO FINAL — EDA NIVEL 1')
print('=' * 70)
print(df_semaforo[['archivo','filas','columnas','pct_nulos',
                    'duplicados_clave','estado']].to_string(index=False))

print('\n\nDECISIONES PARA EL ETL (completar según tus hallazgos):')
print('-' * 70)
decisiones = [
    ('olist_orders',    'Filtrar solo status = delivered'),
    ('olist_items',     'Verificar órdenes sin ítems antes del join'),
    ('olist_payments',  'Descartar columnas de pago — reemplazar con Faker AR'),
    ('olist_products',  'Mapear categorías PT → ES en ETL'),
    ('product_translation', 'Usarla como lookup en el ETL'),
    ('superstore',      'Extraer márgenes por categoría para FactPreciosComp'),
    ('ipc_indec',       '← COMPLETAR: fila de encabezado = ?'),
    ('tipo_cambio',     '← COMPLETAR: nombre de columna de fecha = ?'),
    ('reviews',         'Opcional: agregar score promedio a DimProducto'),
]
for archivo, decision in decisiones:
    print(f'  {archivo:<25} → {decision}')

print('\n' + '=' * 70)
print('PRÓXIMO PASO: EDA Nivel 2 — exploración por grupo funcional')
print('Ahí unimos las tablas y analizamos la coherencia entre ellas')
print('=' * 70)

SEMÁFORO FINAL — EDA NIVEL 1
                                  archivo  filas  columnas  pct_nulos  duplicados_clave                       estado
                 001_olist_orders_dataset  99441         8       0.62                 0             ⚠ Issues menores
        002_olist_order_items_dataset.csv 112650         7       0.00             13984                    ✗ Revisar
     003_olist_order_payments_dataset.csv 103886         5       0.00              4446                    ✗ Revisar
           004_olist_products_dataset.csv  32951         9       0.83                 0             ⚠ Issues menores
005_product_category_name_translation.csv     71         2       0.00                 0                         ✓ OK
              006_Sample - Superstore.csv   9994        21       0.00              4985                    ✗ Revisar
                 007_sh_ipc_aperturas.xls    298       112      10.73                 0 ⚠ Revisar fila de encabezado
       008_tipos-de-cambio-historic